# Rigid Stage — Final Validation Audit and Handoff

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
import json, os, queue, subprocess, sys, threading, time

ROOT = Path('/content/drive/MyDrive/NeuroML_DLIR')
REPO = Path('/content/dlir-partial-fov-main')
DATA = Path('/content/dlir_data')
EVAL = ROOT / 'outputs/rigid_dev_20261021/full_validation_1080'
SCREEN = EVAL / 'similarity_screen'
REFINE = EVAL / 'header_refinement'
OUT = EVAL / 'final_handoff_v1'
BUNDLE = ROOT / 'rigid_stage_release_v1.zip'
MANIFEST = EVAL / 'manifest.json'
OUT.mkdir(parents=True, exist_ok=True)

def run_live(command, label, cwd=None, interval=20):
    start = time.monotonic()
    print(f'[{datetime.now():%H:%M:%S}] {label} started', flush=True)
    p = subprocess.Popen(command, cwd=cwd, env=dict(os.environ, PYTHONUNBUFFERED='1'),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, errors='replace')
    q = queue.Queue()
    def read():
        for line in p.stdout:
            q.put(line)
        q.put(None)
    threading.Thread(target=read, daemon=True).start()
    try:
        while True:
            try:
                item = q.get(timeout=interval)
            except queue.Empty:
                print(f'[{datetime.now():%H:%M:%S}] {label}: {time.monotonic()-start:.0f}s elapsed', flush=True)
                continue
            if item is None:
                break
            print(item, end='', flush=True)
        status = p.wait()
    except KeyboardInterrupt:
        p.terminate()
        raise
    if status:
        raise subprocess.CalledProcessError(status, command)
    print(f'[{datetime.now():%H:%M:%S}] {label} completed in {time.monotonic()-start:.1f}s', flush=True)

assert BUNDLE.is_file(), BUNDLE
assert MANIFEST.is_file(), MANIFEST
assert (REFINE / 'cases.jsonl').is_file()
assert (SCREEN / 'cases.jsonl').is_file()
print('Saved validation results found; no registration will be rerun')

## Environment

In [ ]:
run_live([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'nibabel', 'scipy', 'matplotlib', 'wandb', 'pytest'], 'Dependencies')
if not (REPO / 'scripts/slabsim/evaluation.py').is_file():
    archives = sorted(ROOT.glob('dlir-partial-fov*.zip'))
    assert archives, 'Original repository ZIP missing'
    with ZipFile(archives[0]) as z:
        z.extractall('/content')
assert (REPO / 'scripts/slabsim/evaluation.py').is_file()
if not (DATA / 'test').is_dir():
    archive = ROOT / 'dlir_data.tar.gz'
    assert archive.is_file(), archive
    run_live(['tar', '-xzf', str(archive), '-C', '/content'], 'Restore HCP validation data')
if not (REPO / 'scripts/dl/rigid_stage_handoff.py').is_file():
    assert BUNDLE.is_file(), BUNDLE
    with ZipFile(BUNDLE) as z:
        safe = [s for s in z.namelist() if s.startswith(('scripts/', 'tests/', 'docs/'))]
        assert safe and all('..' not in Path(s).parts for s in safe)
        z.extractall(REPO, members=safe)
run_live([sys.executable, '-m', 'pytest', '-q', 'tests/test_rigid_stage_release.py'], 'Release tests', cwd=REPO)
manifest = json.loads(MANIFEST.read_text())
assert manifest['data_split'] == 'validation'
cases = manifest['cases']
assert len(cases) == 1080 and len({c['relative_dir'] for c in cases}) == 1080
assert len({c['subject'] for c in cases}) == 30
params_files = list((DATA / 'test').glob('*/t*/params.json'))
assert len(params_files) == 1080
if not all(all(k in json.loads(f.read_text()) for k in ('pe_axis','pe_sign','trt')) for f in params_files):
    patch = ROOT / 'dlir_data_params_patch.tar.gz'
    assert patch.is_file(), patch
    run_live(['tar', '-xzf', str(patch), '-C', '/content'], 'Restore acquisition parameters')
assert all(all(k in json.loads(f.read_text()) for k in ('pe_axis','pe_sign','trt')) for f in params_files)
print('Locked validation:', len(cases), 'cases / 30 subjects')

## Paired audit

In [ ]:
AUDIT = OUT / 'audit'
run_live([sys.executable, '-u', 'scripts/dl/audit_rigid_v1.py',
          '--manifest', str(MANIFEST),
          '--header', str(SCREEN / 'cases.jsonl'),
          '--refined', str(REFINE / 'cases.jsonl'),
          '--output', str(AUDIT)], 'Paired validation audit', cwd=REPO)
summary = json.loads((AUDIT / 'summary.json').read_text())
assert summary['cases'] == 1080 and summary['subjects'] == 30
assert summary['failure_count'] == 102
print('Worst-case table:', AUDIT / 'worst_cases.csv')

## Deformable-stage export

In [ ]:
HANDOFF = OUT / 'handoff'
run_live([sys.executable, '-u', 'scripts/dl/rigid_stage_handoff.py',
          '--data', str(DATA), '--manifest', str(MANIFEST),
          '--results', str(REFINE / 'cases.jsonl'),
          '--transforms', str(REFINE / 'transforms'),
          '--output', str(HANDOFF)], 'Rigid transform and PE-metadata export', cwd=REPO)
assert json.loads((HANDOFF / 'handoff_summary.json').read_text())['n_cases'] == 1080

In [ ]:
run_live([sys.executable, '-u', 'scripts/dl/rigid_stage_qc.py',
          '--data', str(DATA),
          '--worst-cases', str(AUDIT / 'worst_cases.csv'),
          '--handoff', str(HANDOFF),
          '--output', str(OUT / 'qc_worst'), '--top', '8'], 'Worst-case T1/EPI overlays', cwd=REPO)

## Outputs

In [ ]:
import wandb
assert wandb.login(), 'W&B online authentication required'
tracking = OUT / 'wandb_audit_complete.json'
if tracking.is_file():
    print('W&B audit already logged:', json.loads(tracking.read_text())['url'])
else:
    runid_path = OUT / 'wandb_run_id.txt'
    runid = runid_path.read_text().strip() if runid_path.is_file() else wandb.util.generate_id()
    run = wandb.init(project='neuroml-dlir', name='rigid-header-refine-handoff-val1080-v1',
                     id=runid, resume='allow', mode='online', dir=str(OUT),
                     config={'split':'validation', 'cases':1080, 'method':'frozen-header-initialized-rigid-refinement-v1'})
    assert run and run.settings.mode == 'online'
    if not runid_path.is_file():
        runid_path.write_text(run.id + '\n')
    for key in ('median_refined_tre_mm', 'p90_refined_tre_mm', 'p95_refined_tre_mm',
                'failure_pct', 'tre_below_2mm_pct', 'paired_improved_pct'):
        run.summary[key] = summary[key]
    artifact = wandb.Artifact('rigid-stage-val1080-audit-v1', type='evaluation',
                              metadata={'cases':1080, 'split':'validation', 'method':'header-refinement-v1'})
    for name in ('summary.json', 'paired_cases.csv', 'subgroups.csv', 'worst_cases.csv', 'regressions.csv'):
        artifact.add_file(str(AUDIT / name), name=name)
    run.log_artifact(artifact)
    url = run.url
    run.finish()
    tracking.write_text(json.dumps({'id':runid,'url':url}, indent=2))
    print('W&B:', url)

package = OUT / 'rigid_to_deformable_handoff_v1.zip'
with ZipFile(package, 'w', compression=ZIP_DEFLATED) as z:
    for f in sorted(HANDOFF.rglob('*')):
        if f.is_file():
            z.write(f, f.relative_to(HANDOFF))
print('Handoff:', package, package.stat().st_size, 'bytes')
print('Audit:', AUDIT / 'summary.json')
print('QC:', OUT / 'qc_worst')
print('This notebook did not train a network or rerun registration.')